# Model 2 - Skill to ESCO matcher

**Task:** unsupervised semantic retrieval / ranking. Input = one free-text course skill
string, output = ranked ESCO concepts + similarity. There are **no gold course-ESCO
labels**, so we (a) build the retriever and (b) evaluate it with the held-out **altLabel**
trick (MODEL_PLAN.md section 5).

Tiers: exact match (baseline) -> char+word TF-IDF cosine -> sentence-transformer embeddings.

> **ESCO source is a group decision (blocker in docs).** Toggle `ESCO_SOURCE` below:
> - `"full_filtered"` = skills_en.csv restricted to transversal + cross-sector (the plan's 4,241).
> - `"clean"` = skills_clean.csv as committed (2,154, no transversal level).
>
> Requires: pandas, numpy, scikit-learn (+ sentence-transformers for the embedding tier).

In [ ]:
import sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd

here = Path.cwd()
REPO_ROOT = here if (here / "data" / "processed").exists() else here.parent
sys.path.insert(0, str(REPO_ROOT))

from src.data_contract import (DATA_PROCESSED, COURSES_CSV,
                               ESCO_SKILLS_FULL_CSV, ESCO_SKILLS_CLEAN_CSV, SEED)
np.random.seed(SEED)

# ---- CONFIG: which ESCO file backs the index (see blocker) ----
ESCO_SOURCE = "full_filtered"   # or "clean"
KEEP_REUSE = {"transversal", "cross-sector"}   # used only for full_filtered
print("repo root:", REPO_ROOT, "| ESCO_SOURCE =", ESCO_SOURCE)

## 1. Build the ESCO concept index

In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

if ESCO_SOURCE == "full_filtered":
    esco = pd.read_csv(ESCO_SKILLS_FULL_CSV, low_memory=False)
    esco = esco[esco["reuseLevel"].isin(KEEP_REUSE)].copy()
elif ESCO_SOURCE == "clean":
    esco = pd.read_csv(ESCO_SKILLS_CLEAN_CSV).copy()
else:
    raise ValueError(ESCO_SOURCE)

esco = esco.dropna(subset=["preferredLabel"]).reset_index(drop=True)
esco["pref_norm"] = esco["preferredLabel"].map(norm)

def alt_list(cell):
    if pd.isna(cell):
        return []
    return [norm(a) for a in str(cell).split("\n") if a.strip()]

esco["alts"] = esco["altLabels"].map(alt_list)
print(f"ESCO concepts in index: {len(esco):,}")
esco[["preferredLabel", "reuseLevel"]].head() if "reuseLevel" in esco else esco[["preferredLabel"]].head()

## 2. Course skill vocabulary

The real product input: the distinct skill strings courses actually use.

In [ ]:
courses = pd.read_csv(COURSES_CSV, low_memory=False)
vocab = set()
for cell in courses["Skills"].dropna():
    for part in str(cell).replace("\n", ",").split(","):
        p = norm(part)
        if p:
            vocab.add(p)
course_vocab = sorted(vocab)
print(f"distinct course skill strings: {len(course_vocab):,}")

## 3. Baseline - exact string match

The floor. Reproduces the 'why not a lookup table' finding: exact match links only a
few percent of course strings to ESCO. It also scores ~0 on the paraphrase test below,
by construction (it cannot retrieve a phrasing it has never seen).

In [ ]:
esco_label_set = set(esco["pref_norm"]) | {a for alts in esco["alts"] for a in alts}
matched = sum(1 for v in course_vocab if v in esco_label_set)
exact_rate = matched / len(course_vocab)
print(f"exact-matched course strings: {matched:,} / {len(course_vocab):,} = {exact_rate:.1%}")
print(f"unmatched: {1-exact_rate:.1%}")

## 4. Held-out altLabel evaluation set

For every concept with an alt label, hold ONE out as a query whose correct answer we
know, and index the concept by its preferred label (+ remaining alts). This yields
thousands of labelled paraphrase queries at zero annotation cost (section 5).

In [ ]:
rng = np.random.default_rng(SEED)
queries, gold = [], []          # query text, correct concept row-index
doc_texts = []                  # one document string per concept (index side)

for i, row in esco.iterrows():
    alts = row["alts"]
    held = None
    if alts:
        held = alts[rng.integers(len(alts))]
    remaining = [a for a in alts if a != held]
    doc_texts.append(" . ".join([row["pref_norm"], *remaining]))
    if held:
        queries.append(held)
        gold.append(i)

print(f"index documents: {len(doc_texts):,} | held-out queries: {len(queries):,}")

In [ ]:
def rank_metrics(sim_matrix, gold_idx, k=5):
    # sim_matrix: (n_queries, n_docs). gold_idx: list of correct doc indices.
    top1 = top5 = mrr = 0.0
    n = len(gold_idx)
    order = np.argsort(-sim_matrix, axis=1)[:, :k]
    for q, g in enumerate(gold_idx):
        ranked = order[q]
        if ranked[0] == g:
            top1 += 1
        if g in ranked:
            top5 += 1
            mrr += 1.0 / (list(ranked).index(g) + 1)
    return {"top1": round(top1/n, 4), "top5": round(top5/n, 4), "mrr5": round(mrr/n, 4)}

eval_results = {}

### 4a. Exact-match on the paraphrase queries (expected ~0)

In [ ]:
pref_lookup = {}
for i, p in enumerate(esco["pref_norm"]):
    pref_lookup.setdefault(p, i)
hit = sum(1 for q, g in zip(queries, gold) if pref_lookup.get(q) == g)
eval_results["exact_match"] = {"top1": round(hit/len(queries), 4), "top5": None, "mrr5": None}
print("exact-match top-1 on paraphrases:", eval_results["exact_match"]["top1"],
      "(near zero by construction - that IS the finding)")

### 4b. Char + word n-gram TF-IDF cosine (no downloads)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)

Dw = word_vec.fit_transform(doc_texts); Qw = word_vec.transform(queries)
Dc = char_vec.fit_transform(doc_texts); Qc = char_vec.transform(queries)

# Combine word + char similarity (simple average).
sim = 0.5 * cosine_similarity(Qw, Dw) + 0.5 * cosine_similarity(Qc, Dc)
eval_results["tfidf_cosine"] = rank_metrics(sim, gold)
print("TF-IDF cosine:", eval_results["tfidf_cosine"])

### 4c. Sentence-transformer embeddings (optional - needs a download)

Guarded so the notebook still runs without the package. Uncomment/install to enable.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    D = model.encode(doc_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    Q = model.encode(queries, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    sim_emb = Q @ D.T
    eval_results["embeddings"] = rank_metrics(sim_emb, gold)
    print("embeddings:", eval_results["embeddings"])
except Exception as e:
    print("Skipped embedding tier:", type(e).__name__, e)

In [ ]:
pd.DataFrame(eval_results).T

## 5. Behaviour on the real course vocabulary

The paraphrase test measures the method; this measures the product. For each course
skill string, take the best ESCO match and look at the similarity distribution and a
few concrete good/bad examples.

In [ ]:
Vw = word_vec.transform(course_vocab); Vc = char_vec.transform(course_vocab)
vsim = 0.5 * cosine_similarity(Vw, Dw) + 0.5 * cosine_similarity(Vc, Dc)
best = vsim.argmax(axis=1); best_score = vsim.max(axis=1)

THRESHOLD = 0.35
cleared = float((best_score >= THRESHOLD).mean())
print(f"share of course strings clearing similarity >= {THRESHOLD}: {cleared:.1%}")

examples = pd.DataFrame({
    "course_skill": course_vocab,
    "best_esco": esco["preferredLabel"].values[best],
    "score": np.round(best_score, 3),
}).sort_values("score", ascending=False)
print("\nTop matches:"); display(examples.head(8))
print("\nWeak matches:"); display(examples.tail(8))

## 6. Write results to files

In [ ]:
out = {
    "esco_source": ESCO_SOURCE,
    "esco_index_size": int(len(esco)),
    "distinct_course_skill_strings": len(course_vocab),
    "exact_match_rate_on_vocab": round(exact_rate, 4),
    "paraphrase_eval": eval_results,
    "course_vocab_threshold": THRESHOLD,
    "course_vocab_cleared_share": round(cleared, 4),
}
(DATA_PROCESSED / "model2_results.json").write_text(
    json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
examples.to_csv(DATA_PROCESSED / "model2_match_examples.csv", index=False, encoding="utf-8")
print("Wrote model2_results.json and model2_match_examples.csv to", DATA_PROCESSED)